# EEGMAT raw-data download and validation

Run this notebook once before the clean preprocessing and baseline notebook.
It downloads EEGMAT v1.0.0 from PhysioNet into Google Drive and verifies that
all 36 subjects have both the rest and arithmetic EDF recordings.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import re
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/URV_Datasets/eegmat/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

expected_pattern = re.compile(r"Subject(\d+)_(1|2)\.edf")
existing = [p for p in RAW_DIR.rglob("*.edf") if expected_pattern.fullmatch(p.name)]

if len(existing) == 72:
    print("All 72 EEGMAT EDF files are already present; download skipped.")
else:
    print(f"Found {len(existing)}/72 EEGMAT EDF files. Downloading/resuming...")
    get_ipython().system(
        "wget -r -N -c -np "
        "https://physionet.org/files/eegmat/1.0.0/ "
        "-P /content/drive/MyDrive/URV_Datasets/eegmat/raw"
    )

## Validate the downloaded dataset

In [ ]:
edf_files = sorted(
    p for p in RAW_DIR.rglob("*.edf") if expected_pattern.fullmatch(p.name)
)

records = []
for path in edf_files:
    match = expected_pattern.fullmatch(path.name)
    records.append((int(match.group(1)), int(match.group(2)), path))

assert len(records) == 72, f"Expected 72 EDF files, found {len(records)}"
subjects = sorted({subject for subject, _, _ in records})
assert subjects == list(range(1, 37)), f"Unexpected subject IDs: {subjects}"

for subject in subjects:
    conditions = sorted(condition for s, condition, _ in records if s == subject)
    assert conditions == [1, 2], f"Subject {subject} conditions: {conditions}"

print("EEGMAT raw-data validation passed.")
print("Subjects:", len(subjects))
print("EDF files:", len(records))
print("Conditions per subject: rest (1) and arithmetic (2)")
print("Raw-data directory:", RAW_DIR)
print("Next: run 01_EEGMAT_Preprocessing_Clean_Utility_Privacy_Baselines.ipynb")